# TeamBoard Webhook Plugin API — Example

This notebook walks through the full lifecycle of a webhook registration:
registering, testing, inspecting deliveries, rotating the secret, and deleting.

**Prerequisites:** `make up && make seed` — the stack must be running locally.

## Setup

In [5]:
import hashlib
import hmac
import json
import time
import requests

BASE_URL = "http://localhost:80"

# Seed credentials (from: make seed)
EMAIL    = "alice@teamboard.local"
PASSWORD = "AliceSecret123!"

## 1. Authenticate

In [6]:
resp = requests.post(f"{BASE_URL}/api/v1/auth/login", json={
    "email": EMAIL,
    "password": PASSWORD,
})
resp.raise_for_status()

access_token = resp.json()["data"]["access_token"]
headers = {"Authorization": f"Bearer {access_token}"}
print("Logged in, token acquired.")

Logged in, token acquired.


## 2. Pick a project

In [7]:
resp = requests.get(f"{BASE_URL}/api/v1/projects", headers=headers)
resp.raise_for_status()

projects = resp.json()["data"]
project_id = projects[0]["id"]
print(f"Using project: {projects[0]['name']}  ({project_id})")

Using project: A  (f4664038-f375-47c6-b246-f1955941d127)


## 3. Register a webhook

**Event filter patterns**
- `"*"` — every event
- `"task.*"` — any event whose type starts with `task.`
- `"task.created"` — exact type match

In [8]:
resp = requests.post(
    f"{BASE_URL}/api/v1/projects/{project_id}/webhooks",
    headers=headers,
    json={
        "target_url": "https://webhook.site/YOUR-UNIQUE-ID",  # replace with your URL
        "description": "Example webhook — task and member events",
        "event_filter": ["task.*", "project.member.added"],
    },
)
resp.raise_for_status()

data       = resp.json()["data"]
webhook_id = data["id"]
secret     = data["secret"]  # returned only once — store it now

print(json.dumps(data, indent=2))
print(f"\nwebhook_id : {webhook_id}")
print(f"secret     : {secret}")

{
  "id": "28b31bf1-1ba6-4ffd-bbc7-4f22cc9e578c",
  "project_id": "f4664038-f375-47c6-b246-f1955941d127",
  "target_url": "https://webhook.site/YOUR-UNIQUE-ID",
  "description": "Example webhook \u2014 task and member events",
  "event_filter": [
    "task.*",
    "project.member.added"
  ],
  "active": true,
  "created_by": "bd15f290-daaf-495d-b6f1-59f35eb2b291",
  "created_at": "2026-06-15T12:47:31.759743Z",
  "updated_at": "2026-06-15T12:47:31.759743Z",
  "secret": "0095808c4960b168d9f88a2d0f8884f7c2278bb5edc640570d91449a00a7f5e5"
}

webhook_id : 28b31bf1-1ba6-4ffd-bbc7-4f22cc9e578c
secret     : 0095808c4960b168d9f88a2d0f8884f7c2278bb5edc640570d91449a00a7f5e5


## 4. Verify an incoming delivery

TeamBoard signs every POST it sends to your endpoint with:

```
HMAC-SHA256(key=secret, msg="{timestamp}.{body}")
```

Headers sent with every delivery:
- `X-TeamBoard-Signature: sha256=<hex>`
- `X-TeamBoard-Timestamp: <unix_seconds>`

In [9]:
def verify_signature(secret: str, body: bytes, signature: str, timestamp: str) -> bool:
    """Return True if the delivery signature is valid."""
    msg = f"{timestamp}.".encode() + body
    expected = "sha256=" + hmac.new(secret.encode(), msg, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, signature)


# Simulate what your webhook endpoint receives:
body      = b'{"event_type":"task.created","data":{"id":"abc"}}'
timestamp = str(int(time.time()))

# Compute signature the same way TeamBoard does:
msg      = f"{timestamp}.".encode() + body
sig      = "sha256=" + hmac.new(secret.encode(), msg, hashlib.sha256).hexdigest()

print("Signature valid:", verify_signature(secret, body, sig, timestamp))

Signature valid: True


## 5. Send a test delivery

In [10]:
resp = requests.post(
    f"{BASE_URL}/api/v1/webhooks/{webhook_id}/test",
    headers=headers,
)
resp.raise_for_status()
print(json.dumps(resp.json(), indent=2))

{
  "data": {
    "delivery_id": "80e9ca1a-60aa-4bb3-b82d-2b4001d74b88"
  }
}


## 6. List deliveries

In [11]:
resp = requests.get(
    f"{BASE_URL}/api/v1/webhooks/{webhook_id}/deliveries",
    headers=headers,
)
resp.raise_for_status()

deliveries = resp.json()["data"]
print(f"{len(deliveries)} delivery/ies found")
for d in deliveries:
    print(f"  {d['id']}  status={d['status']}  attempts={d['attempt_count']}  http={d.get('last_response_status')}")

1 delivery/ies found
  80e9ca1a-60aa-4bb3-b82d-2b4001d74b88  status=dead  attempts=1  http=None


## 7. Inspect a single delivery

In [12]:
if deliveries:
    delivery_id = deliveries[0]["id"]
    resp = requests.get(
        f"{BASE_URL}/api/v1/webhooks/{webhook_id}/deliveries/{delivery_id}",
        headers=headers,
    )
    resp.raise_for_status()
    print(json.dumps(resp.json()["data"], indent=2))

{
  "id": "80e9ca1a-60aa-4bb3-b82d-2b4001d74b88",
  "webhook_id": "28b31bf1-1ba6-4ffd-bbc7-4f22cc9e578c",
  "event_id": "test-393c8092-9f6b-403f-9e06-8e578e56188f",
  "event_type": "webhook.test",
  "status": "dead",
  "attempt_count": 1,
  "last_error": "HTTP 404",
  "last_attempted_at": "2026-06-15T12:47:54.264639Z",
  "failed_permanently_at": "2026-06-15T12:47:54.264639Z",
  "created_at": "2026-06-15T12:47:53.777093Z"
}


## 8. Update the webhook (change filter and description)

In [ ]:
resp = requests.patch(
    f"{BASE_URL}/api/v1/webhooks/{webhook_id}",
    headers=headers,
    json={
        "description": "Updated all events",
        "event_filter": ["*"],
    },
)
resp.raise_for_status()
print(json.dumps(resp.json()["data"], indent=2))

{
  "id": "28b31bf1-1ba6-4ffd-bbc7-4f22cc9e578c",
  "project_id": "f4664038-f375-47c6-b246-f1955941d127",
  "target_url": "https://webhook.site/YOUR-UNIQUE-ID",
  "description": "Updated \u2014 all events",
  "event_filter": [
    "*"
  ],
  "active": true,
  "created_by": "bd15f290-daaf-495d-b6f1-59f35eb2b291",
  "created_at": "2026-06-15T12:47:31.759743Z",
  "updated_at": "2026-06-15T12:48:09.528119Z"
}


## 9. Disable / enable

In [14]:
requests.post(f"{BASE_URL}/api/v1/webhooks/{webhook_id}/disable", headers=headers).raise_for_status()
print("Webhook disabled.")

requests.post(f"{BASE_URL}/api/v1/webhooks/{webhook_id}/enable", headers=headers).raise_for_status()
print("Webhook re-enabled.")

Webhook disabled.
Webhook re-enabled.


## 10. Rotate the secret

Update your receiving server before rotating — the old secret stops working immediately.

In [15]:
resp = requests.post(
    f"{BASE_URL}/api/v1/webhooks/{webhook_id}/rotate-secret",
    headers=headers,
)
resp.raise_for_status()

new_secret = resp.json()["data"]["secret"]
secret = new_secret  # update local reference
print(f"New secret: {new_secret}")

New secret: 25c3d00c6dcd94132f03c5316f44e13a76094a297efa327ebd6a8f8bfacf4cd5


## 11. List all webhooks for the project

In [16]:
resp = requests.get(
    f"{BASE_URL}/api/v1/projects/{project_id}/webhooks",
    headers=headers,
)
resp.raise_for_status()
print(json.dumps(resp.json()["data"], indent=2))

[
  {
    "id": "28b31bf1-1ba6-4ffd-bbc7-4f22cc9e578c",
    "project_id": "f4664038-f375-47c6-b246-f1955941d127",
    "target_url": "https://webhook.site/YOUR-UNIQUE-ID",
    "description": "Updated \u2014 all events",
    "event_filter": [
      "*"
    ],
    "active": true,
    "created_by": "bd15f290-daaf-495d-b6f1-59f35eb2b291",
    "created_at": "2026-06-15T12:47:31.759743Z",
    "updated_at": "2026-06-15T12:48:22.996452Z"
  }
]


## 12. Delete the webhook

In [17]:
resp = requests.delete(
    f"{BASE_URL}/api/v1/webhooks/{webhook_id}",
    headers=headers,
)
resp.raise_for_status()
print(f"Deleted webhook {webhook_id}.")

Deleted webhook 28b31bf1-1ba6-4ffd-bbc7-4f22cc9e578c.
